In [109]:
import pandas as pd
import pickle

df  = pd.read_parquet("data.parquet")

# Chargement du dataset
X_test  = pd.read_parquet("X_test.parquet")
y_test  = pd.read_parquet("y_test.parquet")
X_train  = pd.read_parquet("X_train.parquet")
y_train  = pd.read_parquet("y_train.parquet")

# Chargement des catégories
with open("categories.pkl", "rb") as f:
    categories = pickle.load(f)

# stockage des metriques
global_metrics = []

# Fonctions de stockage des résultats

In [110]:
import pickle
from pathlib import Path

def load_results(results_path):
    if results_path.exists():
        # sauvegarde également les différentes catègories
        with open(results_path, "rb") as f:
            return pickle.load(f)
    else:
        return pd.DataFrame()
        
def save_results(results, results_path):
    # sauvegarde également les différentes catègories
    with open(results_path, "wb") as f:
        pickle.dump(results, f)

# Fonction de test

In [111]:
from collections.abc import Callable
import time
import pandas as pd


def test(
    indices,
    model,
    **kwargs
) -> pd.DataFrame:
    """
    Évalue les performances du modèle de classification sur un ensemble
    d'indices du jeu de test.

    Pour chaque réclamation, la fonction construit éventuellement un prompt
    complémentaire, puis effectue la classification via `with_retry`.
    Le prompt complémentaire peut être fourni directement sous forme de
    chaîne de caractères ou être généré dynamiquement par une fonction.

    Args:
        indices:
            Collection d'indices correspondant aux lignes de `X_test` et
            `y_test` à utiliser pour l'évaluation.

        additional_prompt (str | Callable[[str], str] | None, optional):
            Prompt complémentaire à ajouter au prompt de classification.

            Trois comportements sont possibles :

            - `None` : aucun prompt complémentaire.
            - `str` : le même prompt est utilisé pour toutes les questions.
            - `Callable[[str], str]` : la fonction est appelée pour chaque
              question avec le texte de la réclamation comme argument.
              Elle doit retourner le prompt complémentaire à utiliser.

    Returns:
        pd.DataFrame:
            Tableau récapitulatif contenant les résultats de chaque test.

            Colonnes :
            - `Index` : index de la réclamation.
            - `Question` : réclamation testée.
            - `Réponse` : catégorie prédite.
            - `Attendue` : catégorie réelle.
            - `Correct` : indique si la prédiction est correcte.
            - `Temps (s)` : temps nécessaire pour obtenir la réponse.

    Raises:
        TypeError:
            Si `additional_prompt` n'est ni `None`, ni une chaîne de
            caractères, ni une fonction appelable.

    Example:
        # Prompt fixe
        results = test(
            indices,
            additional_prompt="Soyez particulièrement attentif..."
        )

        # Prompt généré dynamiquement
        def get_examples(question):
            examples = retrieve_examples(
                metadata,
                question,
                k=3
            )

            return format_examples(examples)

        results = test(
            indices,
            additional_prompt=get_examples
        )
    """

    results = []


    for idx in indices:
        X = X_test.loc[idx]
        y = y_test.loc[idx]

        question = X
        expected = y

        start = time.perf_counter()

        y_pred = model.predict([X])

        elapsed = time.perf_counter() - start

        results.append({
            "Index": idx,
            "Question": question,
            "Réponse": y_pred[0],
            "Attendue": expected,
            "Correct": y_pred[0] == expected,
            "Temps (s)": elapsed
        })

    return pd.DataFrame(results)

# Test 1
À partir du texte d'une réclamation, prédire automatiquement son Tag

l'objectif est de déterminer si le contenu de la réclamation permet lui-même d'identifier la catégorie.

In [112]:
# dans cette approche, on prédit le Tag à partir du texte de réclamation
# Cela permet d'avoir une comparaison avec l'approche LLM
filter = (X_test["Consumer Claim"].isna() == False & X_test["Consumer Claim"].str.strip().ne(""))
X_test  = X_test[filter]["Consumer Claim"]
y_test  = y_test[filter]["Tag"]

filter = (X_train["Consumer Claim"].isna() == False & X_train["Consumer Claim"].str.strip().ne(""))
X_train  = X_train[filter]["Consumer Claim"]
y_train  = y_train[filter]["Tag"]

In [113]:
X_test.head(10)

192075    I generally let people walk over me you could ...
131727    MR. XXXX calls and tells me he is with the leg...
455266    I am including my marriage license per your re...
80575     Disputed with company on XX/XX/XXXX. The compa...
551360    On XXXX XXXX, XXXX, we turned-over our XXXX XX...
694534    Transunion deleted XXXX XXXX and XXXX. I have ...
776888    I contacted CFPB two years ago about how Bayvi...
192053    We have received multiple calls from Commerica...
721113    XXXX alleged that I owe them {$76.00}. for a p...
49478     To whom it may concern On XX/XX/XXXX Radius Gl...
Name: Consumer Claim, dtype: str

In [114]:
y_test.head(10)

192075    Credit reporting, credit repair services, or o...
131727                                      Debt collection
455266    Credit reporting, credit repair services, or o...
80575                                       Debt collection
551360            Payday loan, title loan, or personal loan
694534    Credit reporting, credit repair services, or o...
776888                                             Mortgage
192053                                      Debt collection
721113                                      Debt collection
49478     Credit reporting, credit repair services, or o...
Name: Tag, dtype: str

## TF-IDF

**TF-IDF est une méthode qui permet de transformer des textes en vecteurs numériques, afin qu'un algorithme de machine learning puisse les exploiter.**

Il est utile pour transformer Consumer Claim en données numériques avant d'être utilisé par un modèle tel que régression-logistique, SVM ou Random Forest.

TF-IDF signifie Term Frequency – Inverse Document Frequency.

L'idée générale est simple : un mot doit avoir un poids élevé s'il est important dans une réclamation, mais un poids faible s'il apparaît dans presque toutes les réclamations.

In [115]:
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

# L'entrainement est réalisé uniquement sur le train
vectorizer_path = Path("tfidf_vectorizer.joblib")
try:
    print(vectorizer_path, ": Charge le modèle existant...")
    vectorizer = joblib.load(vectorizer_path)
except Exception as e:
    print(e)
    print(vectorizer_path, ": Entraine un nouveau modèle... ")
    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )
    vectorizer.fit(X_train)

    # Sauvegarde le modèle entraîné
    joblib.dump(vectorizer, vectorizer_path)

vectorizer.get_feature_names_out()

# transforme les données en vecteur
X_train_tfidf = vectorizer.transform(X_train)

tfidf_vectorizer.joblib : Charge le modèle existant...


## Modèle 1 : TF-IDF + Logistic Regression

In [116]:
from sklearn.linear_model import LogisticRegression

classifier_path = Path("logistic_regression.joblib")
try:
    print(classifier_path, ": Charge le modèle existant...")
    classifier = joblib.load(classifier_path)
except Exception as e:
    print(e)
    print(classifier_path, ": Entraine un nouveau modèle... ")
    classifier = LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    )
    classifier.fit(X_train_tfidf, y_train)
    
    # Sauvegarde le modèle entraîné
    joblib.dump(classifier, classifier_path)

logistic_regression.joblib : Charge le modèle existant...


In [117]:
from pathlib import Path
from sklearn.pipeline import Pipeline
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

name = "Logistic Regression"
method = "LogisticRegression"
results_path = Path(name + ".pkl")
results = load_results(results_path)
 
model = Pipeline([
    ("tfidf", vectorizer),
    ("classifier", classifier)
])

if len(results) == 0:
    print("Démarre le test:", name)
    indices = X_test.index
    results = test(indices, model)
    save_results(results, results_path)

metrics = calculate_metrics(name, method, results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)

global_metrics.append(metrics)


------------------------
Logistic Regression
------------------------
Name : Logistic Regression
Method : LogisticRegression
Samples : 76791
Accuracy : 83.63%
Precision (macro) : 71.37%
Recall (macro) : 74.87%
F1 (macro) : 72.61%
Precision (weighted) : 84.51%
Recall (weighted) : 83.63%
F1 (weighted) : 83.91%
Temps moyen (s) : 0.001 s
Temps médian (s) : 0.001 s
Temps P95 (s) : 0.001 s
------------------------
                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       0.79      0.85      0.82      5530
                                                 Credit card or prepaid card       0.79      0.83      0.81      8446
Credit reporting, credit repair services, or other personal consumer reports       0.91      0.82      0.86     24675
                                                             Debt collection       0.85      0.83      0.84     17

## Modèle 2 : TF-IDF + Linear SVM

In [118]:
from sklearn.svm import LinearSVC

svm_path = Path("svm.joblib")
try:
    print(svm_path, ": Charge le modèle existant...")
    svm = joblib.load(svm_path)
except Exception as e:
    print(e)
    print(svm_path, ": Entraine un nouveau modèle... ")
    svm = LinearSVC(
        C=1.0,
        class_weight="balanced",
        max_iter=5000
    )
    svm.fit(X_train_tfidf, y_train)
    
    # Sauvegarde le modèle entraîné
    joblib.dump(svm, svm_path)

svm.joblib : Charge le modèle existant...


In [119]:
from pathlib import Path
from sklearn.pipeline import Pipeline
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

name = "Linear SVC"
method = "LinearSVC"
results_path = Path(name + ".pkl")
results = load_results(results_path)
 
model = Pipeline([
    ("tfidf", vectorizer),
    ("classifier", svm)
])

if len(results) == 0:
    print("Démarre le test:", name)
    indices = X_test.index
    results = test(indices, model)
    save_results(results, results_path)

metrics = calculate_metrics(name, method, results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)

global_metrics.append(metrics)


------------------------
Linear SVC
------------------------
Name : Linear SVC
Method : LinearSVC
Samples : 76791
Accuracy : 86.36%
Precision (macro) : 78.74%
Recall (macro) : 71.67%
F1 (macro) : 72.73%
Precision (weighted) : 86.27%
Recall (weighted) : 86.36%
F1 (weighted) : 86.28%
Temps moyen (s) : 0.001 s
Temps médian (s) : 0.001 s
Temps P95 (s) : 0.001 s
------------------------
                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       0.81      0.85      0.83      5530
                                                 Credit card or prepaid card       0.81      0.84      0.83      8446
Credit reporting, credit repair services, or other personal consumer reports       0.91      0.89      0.90     24675
                                                             Debt collection       0.86      0.87      0.87     17424
                       

## Modèle 3 : TF-IDF + Naive Bayes

In [120]:
from sklearn.naive_bayes import ComplementNB

nb_path = Path("nb.joblib")
try:
    print(nb_path, ": Charge le modèle existant...")
    nb = joblib.load(nb_path)
except Exception as e:
    print(e)
    print(nb_path, ": Entraine un nouveau modèle... ")
    nb = ComplementNB()
    nb.fit(X_train_tfidf, y_train)
    
    # Sauvegarde le modèle entraîné
    joblib.dump(nb, nb_path)

nb.joblib : Charge le modèle existant...


In [121]:
from pathlib import Path
from sklearn.pipeline import Pipeline
from metrics import (
    calculate_metrics,
    print_metrics,
    report,
)

name = "Complement NB"
method = "ComplementNB"
results_path = Path(name + ".pkl")
results = load_results(results_path)
 
model = Pipeline([
    ("tfidf", vectorizer),
    ("classifier", nb)
])

if len(results) == 0:
    print("Démarre le test:", name)
    indices = X_test.index
    results = test(indices, model)
    save_results(results, results_path)

metrics = calculate_metrics(name, method, results)

print()
print("------------------------")
print(name)
print("------------------------")
print_metrics(metrics)
print("------------------------")
report(results)

global_metrics.append(metrics)


------------------------
Complement NB
------------------------
Name : Complement NB
Method : ComplementNB
Samples : 76791
Accuracy : 80.24%
Precision (macro) : 70.96%
Recall (macro) : 56.35%
F1 (macro) : 58.31%
Precision (weighted) : 79.80%
Recall (weighted) : 80.24%
F1 (weighted) : 78.52%
Temps moyen (s) : 0.030 s
Temps médian (s) : 0.029 s
Temps P95 (s) : 0.032 s
------------------------
                                                                              precision    recall  f1-score   support

                                                 Checking or savings account       0.78      0.76      0.77      5530
                                                 Credit card or prepaid card       0.79      0.75      0.77      8446
Credit reporting, credit repair services, or other personal consumer reports       0.80      0.91      0.85     24675
                                                             Debt collection       0.82      0.80      0.81     17424
              

# Tableau comparatif synthétique des métriques

In [122]:
global_metrics_df = pd.DataFrame(
    global_metrics,
    columns=["Name", "Samples", "Method", "Accuracy", "F1 (macro)", "Temps moyen (s)"]
    )

global_metrics_df = global_metrics_df.rename(columns={
    "Name" : "Configuration du Test",
    "Samples" : "Échantillon (N)",
    "Method" : "Modèle",
    "Accuracy": "Accuracy",
    "F1-Score": "F1 (macro)",
    "Temps moyen (s)" : "Temps Moyen (s)"
})

global_metrics_df

,Configuration du Test,Échantillon (N),Modèle,Accuracy,F1 (macro),Temps Moyen (s)
0,Logistic Regression,76791,LogisticRegression,0.836309,0.726071,0.000770
1,Linear SVC,76791,LinearSVC,0.863643,0.727310,0.000773
2,Complement NB,76791,ComplementNB,0.802386,0.583090,0.029511
